In [ ]:
import sys
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(name)s: %(message)s')

from src.agents.state import AgentState


def make_state(query):
    return {
        'messages': [], 'current_query': query, 'original_query': query,
        'sub_queries': [], 'research_plan': '',
        'retrieved_documents': [], 'retrieval_attempts': 0,
        'draft_answer': '', 'cited_sources': [],
        'evaluation_score': 0.0, 'evaluation_feedback': '',
        'hallucination_detected': False, 'next_agent': 'researcher',
        'iteration_count': 0, 'is_complete': False, 'error': None,
    }


# ─── 1. Researcher Agent ───
print('═' * 60)
print(' RESEARCHER AGENT')
print('═' * 60)

from src.agents.researcher import ResearcherAgent
researcher = ResearcherAgent()

for q in ["What is RAG?", "Compare transformers and RNNs for handling long sequences"]:
    result = researcher.run(make_state(q))
    print(f'\n  Q: {q}')
    print(f'  Sub-queries: {result["sub_queries"]}')
    print(f'  Plan: {result["research_plan"]}')


# ─── 2. Retriever Agent ───
print(f'\n\n{"═" * 60}')
print(' RETRIEVER AGENT')
print('═' * 60)

from src.agents.retriever_agent import RetrieverAgent
retriever_agent = RetrieverAgent()

state = make_state("What is retrieval augmented generation?")
state['sub_queries'] = ['What is retrieval augmented generation?']
ret_result = retriever_agent.run(state)

print(f'\n  Documents: {len(ret_result["retrieved_documents"])}')
for i, doc in enumerate(ret_result['retrieved_documents'][:3]):
    print(f'  [{i+1}] score={doc["score"]:.4f} | {doc["content"][:60]}...')


# ─── 3. Synthesizer Agent ───
print(f'\n\n{"═" * 60}')
print('  SYNTHESIZER AGENT')
print('═' * 60)

from src.agents.synthesizer import SynthesizerAgent
synthesizer = SynthesizerAgent()

state = make_state("What is RAG and how does it work?")
state['retrieved_documents'] = ret_result['retrieved_documents']
state['research_plan'] = 'Direct answer'
synth_result = synthesizer.run(state)

print(f'\n  Answer ({len(synth_result["draft_answer"])} chars):')
print(f'  {synth_result["draft_answer"][:300]}...')
print(f'  Citations: {synth_result["cited_sources"]}')


# ─── 4. Evaluator Agent ───
print(f'\n\n{"═" * 60}')
print(' EVALUATOR AGENT')
print('═' * 60)

from src.agents.evaluator import EvaluatorAgent
evaluator = EvaluatorAgent()

# Good answer
state = make_state("What is RAG?")
state['retrieved_documents'] = ret_result['retrieved_documents']
state['draft_answer'] = synth_result['draft_answer']
eval_result = evaluator.run(state)

print(f'\n  Good Answer Eval:')
print(f'    Score: {eval_result["evaluation_score"]:.2f}')
print(f'    Hallucination: {eval_result["hallucination_detected"]}')
print(f'    Pass: {eval_result["is_complete"]}')

# Hallucinated answer
bad_state = make_state("What is RAG?")
bad_state['retrieved_documents'] = [{'content': 'RAG uses retrieval for grounding.', 'metadata': {}, 'score': 0.9, 'rank': 1}]
bad_state['draft_answer'] = 'RAG was invented by Elon Musk in 2030 using quantum computers and costs $1 billion per query.'
bad_eval = evaluator.run(bad_state)

print(f'\n  Hallucinated Answer Eval:')
print(f'    Score: {bad_eval["evaluation_score"]:.2f}')
print(f'    Hallucination: {bad_eval["hallucination_detected"]}')
print(f'    Pass: {bad_eval["is_complete"]}')
print(f'    Feedback: {bad_eval["evaluation_feedback"]}')


# ─── 5. Full Graph Trace ───
print(f'\n\n{"═" * 60}')
print(' FULL GRAPH EXECUTION TRACE')
print('═' * 60)

from src.agents.graph import agent_executor

query = "How does hybrid retrieval improve RAG compared to dense-only search?"
state = make_state(query)
config = {'configurable': {'thread_id': 'debug-1'}}

print(f'\n  Query: {query}\n')

for step_num, step in enumerate(agent_executor.stream(state, config)):
    for node_name, node_output in step.items():
        print(f'  Step {step_num+1} → {node_name}')
        for key, val in node_output.items():
            if key in ('retrieved_documents', 'messages'):
                print(f'    {key}: [{len(val) if val else 0} items]')
            elif isinstance(val, str) and len(val) > 80:
                print(f'    {key}: "{val[:80]}..."')
            elif val not in (None, '', [], 0, 0.0, False):
                print(f'    {key}: {val}')

print(f'\n Agent debugging complete')